# 2017–2018 SOXX 首次披露来源

六份精确原文与修订排除边界见 `plan.json`。

In [1]:
import json,hashlib,subprocess,tempfile,tarfile,shlex,time
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd();card=root/'research/stage1_soxx_2017_2018_original_concentration_source_20261023';plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['already_qualified_2018_Sep_numeric_result_sha256']==sha(root/'research/stage1_soxx_2018_sep_concentration_20261023/result.json')
assert plan['frozen'] and plan['series_calendar_sha256']==sha(root/'research/stage1_soxx_series_filing_search_20261023/series_report_calendar_inventory.json')
rawdir=root/'data/snapshots/stage1_soxx_2017_2018_original_concentration_source_20261023';rawdir.mkdir(parents=True,exist_ok=True)
claim=card/'attempt_started.json';out=card/'transport_result.json'
assert not claim.exists() and not out.exists() and not any(rawdir.iterdir())
claim.write_text(json.dumps({'plan_sha256':sha(card/'plan.json'),'started_at_utc':datetime.now(timezone.utc).isoformat(),'maximum_GETs':6},indent=2)+'\n')
requests=[]
for j,item in enumerate(plan['reports']):
 if j:time.sleep(plan['request']['minimum_seconds_between_requests'])
 raw=rawdir/(item['accession']+'_'+item['primary_document']);headers=rawdir/(raw.name+'.headers.txt')
 started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--compressed','--http1.1','--connect-timeout',str(plan['request']['connect_timeout_seconds']),
  '--max-time',str(plan['request']['total_timeout_seconds_each']),'--max-filesize',str(plan['request']['max_response_bytes_each']),
  '--fail-with-body','-sS','-D',str(headers),'-o',str(raw),'-w','%{http_code}\n%{url_effective}',
  '-H','User-Agent: '+plan['request']['user_agent'],'-H','Accept: text/html',item['url']]
 proc=subprocess.run(cmd,capture_output=True,text=True,timeout=plan['request']['total_timeout_seconds_each']+15)
 ended=datetime.now(timezone.utc).isoformat();lines=proc.stdout.splitlines();status=int(lines[0]) if lines and lines[0].isdigit() else None
 entry={'report_date':item['report_date'],'form':item['form'],'accession':item['accession'],'accepted_at_utc':item['accepted_at_utc'],
  'url':item['url'],'requested_at_utc':started,'received_at_utc':ended,'http_status':status,
  'final_url':lines[1] if len(lines)>1 else None,'curl_exit_code':proc.returncode,'curl_stderr':proc.stderr[:400],
  'raw_path':str(raw.relative_to(root)),'raw_sha256':sha(raw) if raw.exists() else None,'raw_bytes':raw.stat().st_size if raw.exists() else 0,
  'headers_sha256':sha(headers) if headers.exists() else None,'headers_bytes':headers.stat().st_size if headers.exists() else 0}
 requests.append(entry)
 if proc.returncode or status!=200 or entry['final_url']!=item['url'] or not raw.exists() or raw.stat().st_size>plan['request']['max_response_bytes_each']:
  entry['stop_reason']='transport_or_identity_failed';break
receipt={'status':'six_original_2017_2018_SEC_bodies_received_identity_pending' if len(requests)==6 and all('stop_reason' not in x for x in requests) else 'original_2017_2018_body_batch_stopped',
 'plan_sha256':sha(card/'plan.json'),'GETs_consumed':len(requests),'requests':requests,'SOXX_schedule_identities_qualified':0,
 'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
out.write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
with tempfile.TemporaryDirectory(prefix='soxx-2017-18-original-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as tar:
  for path in [claim,out,*sorted(rawdir.iterdir())]:
   if path.exists():tar.add(path,arcname=path.name)
 settings=_settings(root);digest=sha(archive)
 evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',
  path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 with tempfile.TemporaryDirectory(prefix='soxx-2017-18-original-restore-') as tmp2:
  copy=Path(tmp2)/'restore.tar.gz'
  with copy.open('wb') as out_file:
   check=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],
    'cat '+shlex.quote(remote)],stdout=out_file,stderr=subprocess.PIPE,timeout=120)
  assert check.returncode==0 and sha(copy)==digest
  with tarfile.open(copy,'r:gz') as stored:
   for entry in requests:
    if entry['raw_sha256'] is None:continue
    member=stored.extractfile(Path(entry['raw_path']).name)
    assert member and hashlib.sha256(member.read()).hexdigest()==entry['raw_sha256']
 receipt['nas_evidence']=evidence;receipt['nas_scoped_restore_verified']=True
 out.write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
print({'status':receipt['status'],'GETs':len(requests),'raw_MB':round(sum(x['raw_bytes'] for x in requests)/1e6,2),'NAS_restore':True})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'six_original_2017_2018_SEC_bodies_received_identity_pending', 'GETs': 6, 'raw_MB': 296.73, 'NAS_restore': True}


In [1]:
import json,hashlib,re
from pathlib import Path
from datetime import datetime
from bs4 import BeautifulSoup
root=Path.cwd();card=root/'research/stage1_soxx_2017_2018_original_concentration_source_20261023';plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
transport=json.loads((card/'transport_result.json').read_text())
assert transport['status']=='six_original_2017_2018_SEC_bodies_received_identity_pending' and transport['nas_scoped_restore_verified']
assert transport['plan_sha256']==sha(card/'plan.json') and len(transport['requests'])==6
assert plan['already_qualified_2018_Sep_numeric_result_sha256']==sha(root/plan['already_qualified_2018_Sep_numeric_result'])
assert not (card/'identity_result.json').exists()
target=plan['target_fund_title'].upper();checks=[]
for item in transport['requests']:
 body=(root/item['raw_path']).read_bytes();assert hashlib.sha256(body).hexdigest()==item['raw_sha256']
 expected=datetime.strptime(item['report_date'],'%Y-%m-%d').strftime('%B %d, %Y')
 selected=None;inline=[];toc=[]
 if item['form']=='N-Q':
  for match in re.finditer(re.escape(target.lower().encode()),body.lower()):
   pos=match.start();frag=body[pos:pos+5000]
   plain=' '.join(BeautifulSoup(frag.decode('latin1',errors='replace'),'html.parser').get_text(' ',strip=True).split())
   accepted=expected in plain[:1400] and all(x in plain[:3000] for x in ('Security','Shares','Value')) and 'COMMON STOCKS' in plain.upper()
   inline.append({'method':'inline_title','offset':pos,'accepted':accepted})
   if accepted:selected=inline[-1];break
 else:
  for match in re.finditer(rb'<A\s+HREF="#([^"]+)"[^>]*>(.*?)</A>',body,re.I|re.S):
   label=' '.join(BeautifulSoup(match.group(2).decode('latin1',errors='replace'),'html.parser').get_text(' ',strip=True).split()).upper()
   if target not in label:continue
   anchor=match.group(1).decode('ascii',errors='ignore')
   found=re.search(rb'NAME="'+re.escape(anchor.encode())+rb'"',body[match.end():],re.I)
   if found is None:continue
   start=match.end()+found.start();next_match=re.search(rb'NAME="toc[^"]+"',body[start+10:],re.I)
   end=(start+10+next_match.start()) if next_match else min(len(body),start+100000)
   section=body[start:end];plain=' '.join(BeautifulSoup(section.decode('latin1',errors='replace'),'html.parser').get_text(' ',strip=True).split())
   accepted=target in plain.upper() and expected in plain and all(x in plain for x in ('Security','Shares','Value')) and 'COMMON STOCKS' in plain.upper()
   toc.append({'method':'TOC_anchor_after_fund_link','anchor':anchor,'fund_link_byte_offset':match.start(),'start':start,'end':end,
    'section_sha256':hashlib.sha256(section).hexdigest(),'accepted':accepted})
   if accepted:selected=toc[-1];break
 checks.append({'report_date':item['report_date'],'form':item['form'],'accession':item['accession'],'accepted_at_utc':item['accepted_at_utc'],
  'raw_sha256':item['raw_sha256'],'raw_bytes':item['raw_bytes'],'inline_hits':inline,'TOC_hits':toc,
  'SOXX_schedule_identity_pass':selected is not None,'selected_identity_method':selected['method'] if selected else None,
  'selected_identity_location':selected})
result={'status':'six_2017_2018_first_original_SOXX_schedule_identities_confirmed_source_only' if all(x['SOXX_schedule_identity_pass'] for x in checks) else '2017_2018_first_original_SOXX_identities_partial_or_unqualified',
 'plan_sha256':sha(card/'plan.json'),'checks':checks,'identity_pass_count':sum(x['SOXX_schedule_identity_pass'] for x in checks),
 'source_NAS_archive_sha256':transport['nas_evidence']['sha256'],'source_NAS_scoped_restore_verified':True,
 '2018_Sep_reused_result_sha256':plan['already_qualified_2018_Sep_numeric_result_sha256'],
 'later_amendment_accessions_excluded':[x['accession'] for x in plan['later_amendments_excluded_from_first_version']],
 'new_HTTP_GETs':0,'security_values_extracted':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,
 'full_2017_2018_numeric_coverage':False}
(card/'identity_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'status':result['status'],'pass_count':result['identity_pass_count'],'methods':[x['selected_identity_method'] for x in checks]})


{'status': 'six_2017_2018_first_original_SOXX_schedule_identities_confirmed_source_only', 'pass_count': 6, 'methods': ['TOC_anchor_after_fund_link', 'inline_title', 'TOC_anchor_after_fund_link', 'inline_title', 'TOC_anchor_after_fund_link', 'inline_title']}


In [1]:
import json,hashlib,re
from pathlib import Path
from datetime import datetime,timezone
from bs4 import BeautifulSoup
root=Path.cwd();card=root/'research/stage1_soxx_2017_2018_original_concentration_source_20261023';plan=json.loads((card/'numeric_2017_2018_plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['source_transport_sha256']==sha(card/'transport_result.json')
assert plan['source_identity_sha256']==sha(card/'identity_result.json')
transport=json.loads((card/'transport_result.json').read_text());identity=json.loads((card/'identity_result.json').read_text())
assert transport['nas_scoped_restore_verified'] and transport['nas_evidence']['sha256']==plan['source_NAS_archive_sha256']
assert identity['identity_pass_count']==6
assert plan['reused_2018_Sep_result_sha256']==sha(root/plan['reused_2018_Sep_result_path'])
claim=card/'numeric_2017_2018_attempt_started.json';out=card/'numeric_2017_2018_result.json'
assert not claim.exists() and not out.exists()
claim.write_text(json.dumps({'plan_sha256':sha(card/'numeric_2017_2018_plan.json'),'started_at_utc':datetime.now(timezone.utc).isoformat(),'raw_report_parses_started':6},indent=2)+'\n')
integer=re.compile(r'^[0-9][0-9,]*$');results=[]
for item in plan['reports']:
 raw=root/item['raw_path'];payload=raw.read_bytes();assert len(payload)==item['raw_bytes'] and hashlib.sha256(payload).hexdigest()==item['raw_sha256']
 a,b=item['bounded_section_byte_range'];section=payload[a:b]
 assert hashlib.sha256(section).hexdigest()==item['bounded_section_sha256']
 rows=BeautifulSoup(section.decode('latin1',errors='replace'),'html.parser').find_all('tr')
 parsed=[];ambiguous=[];started=False;total_label=False;printed=None
 for row in rows:
  cells=[x.get_text(' ',strip=True).replace('\xa0',' ').strip() for x in row.find_all(['td','th'],recursive=False)]
  nonempty=[x for x in cells if x and x!='$']
  if not nonempty:continue
  name=nonempty[0];upper=name.upper();nums=[int(x.replace(',','')) for x in nonempty[1:] if integer.fullmatch(x)]
  if not started:
   if 'COMMON STOCKS' in upper and 'TOTAL' not in upper:started=True
   continue
  if 'TOTAL COMMON STOCKS' in upper:
   total_label=True
   if nums:printed=nums[-1];break
   continue
  if total_label:
   if nums:printed=nums[-1]
   break
  if upper==name and ('—' in name or '–' in name or 'COMMON STOCKS' in upper):continue
  if len(nums)>=2 and any(ch.isalpha() for ch in name):
   parsed.append({'security_original':name,'shares_original':nums[0],'fair_value_usd_original':nums[-1]})
  elif any(ch.isalpha() for ch in name) and nums:
   ambiguous.append({'row_text':nonempty[:5],'numeric_tokens':nums})
 values=[x['fair_value_usd_original'] for x in parsed];total=sum(values)
 for x in parsed:x['reported_equity_weight']=x['fair_value_usd_original']/total if total else None
 checks={'common_stocks_heading_found':started,'total_label_found':total_label,'printed_total_found':printed is not None,
  'row_count_20_to_40':20<=len(parsed)<=40,'all_positive':all(x['shares_original']>0 and x['fair_value_usd_original']>0 for x in parsed),
  'distinct_names':len({x['security_original'] for x in parsed})==len(parsed),
  'total_closure_within_1_usd':printed is not None and abs(total-printed)<=1,'ambiguous_rows_zero':len(ambiguous)==0}
 ranked=sorted(parsed,key=lambda x:x['fair_value_usd_original'],reverse=True)
 results.append({'report_date':item['report_date'],'form':item['form'],'accession':item['accession'],'accepted_at_utc':item['accepted_at_utc'],
  'raw_sha256':item['raw_sha256'],'section_sha256':item['bounded_section_sha256'],
  'status':'numeric_source_closure_passed' if all(checks.values()) else 'numeric_source_unqualified',
  'parsed_equity_rows':len(parsed),'equities':parsed,'ambiguous_rows':ambiguous,
  'parsed_fair_value_total_usd':total,'printed_total_common_stocks_usd':printed,
  'total_closure_difference_usd':total-printed if printed is not None else None,'checks':checks,
  'top5_disclosed_equity_value_share':sum(x['reported_equity_weight'] for x in ranked[:5]) if total else None,
  'herfindahl_sum_squared_disclosed_equity_weights':sum(x['reported_equity_weight']**2 for x in parsed) if total else None})
result={'status':'six_2017_2018_first_original_SOXX_reported_values_closed' if all(x['status']=='numeric_source_closure_passed' for x in results) else '2017_2018_first_original_SOXX_reported_values_partial_or_unqualified',
 'plan_sha256':sha(card/'numeric_2017_2018_plan.json'),'reports':results,'reports_attempted':len(results),
 'reports_with_numeric_closure':sum(x['status']=='numeric_source_closure_passed' for x in results),
 'source_NAS_archive_sha256':transport['nas_evidence']['sha256'],'source_NAS_scoped_restore_verified':True,
 'later_amendments_separate':plan['amendment_boundary'],
 'reused_2018_Sep_result_sha256':plan['reused_2018_Sep_result_sha256'],
 'full_2018_Dec_2022_training_coverage':False,'held_2023_2026_coverage':False,
 'HTTP_GETs_new':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,
 'scientific_boundary':'These six first-filed 2017-2018 values add source coverage without duplicating the separately qualified 2018 September report but do not establish incremental information or a Stage1 0bp account.'}
out.write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'status':result['status'],'per_report':[(x['report_date'],x['parsed_equity_rows'],x['total_closure_difference_usd'],x['status']) for x in results]})


{'status': '2017_2018_first_original_SOXX_reported_values_partial_or_unqualified', 'per_report': [('2017-03-31', 30, 0, 'numeric_source_closure_passed'), ('2017-06-30', 30, 0, 'numeric_source_closure_passed'), ('2017-09-30', 30, 0, 'numeric_source_closure_passed'), ('2017-12-31', 30, 0, 'numeric_source_closure_passed'), ('2018-03-31', 30, 0, 'numeric_source_closure_passed'), ('2018-06-30', 0, None, 'numeric_source_unqualified')]}


## 中文结论

执行后见 `summary.json` 与 `conclusion.md`；来源资格不是 SOXX 超额。